### 📚 复习卡:ReAct 一次调用内部到底发生了什么

以 `((15+23)×64+52)/6` 为例。

#### 1. Graph 结构(只有两个节点)

```
START → agent ──┬──► tools ──► agent   (回环)
                └──► END               (条件边:看最新 AIMessage 是否还有 tool_calls)
```

- **`agent` 节点** = 调用 LLM,产出一个 `AIMessage`
- **`tools` 节点** = `ToolNode`,执行该 AIMessage 里的所有 `tool_calls`(并行)
- **路由决策只发生在 `agent` 之后**:`AIMessage.tool_calls` 非空 → `tools`;为空 → `END`
- `tools → agent` 是**无条件回边**,工具一执行完一定回到 LLM

#### 2. 执行轨迹(13 条 messages)

```
[0]  Human                "Please calculate ((15+23)×64+52)/6?"
[1]  AI    tool_calls=[add(15,23), mul(64,1), add(52,0), div(6,1)]   ← 第1轮:并行4个
[2-5] Tool 38 / 64 / 52 / 6.0                                          ← ToolNode 并发
[6]  AI    tool_calls=[mul(38, 64)]                                    ← 第2轮
[7]  Tool  2432
[8]  AI    tool_calls=[add(2432, 52)]                                  ← 第3轮
[9]  Tool  2484
[10] AI    tool_calls=[div(2484, 6)]                                   ← 第4轮
[11] Tool  414.0
[12] AI    "The result is 414.0"   tool_calls=[]   finish_reason='stop'  ← END
```

> LLM 想"并行",但中间结果有依赖,只对叶子常量并行得起来 —— 第 1 轮 4 个调用里只有 `add(15,23)` 真有用,另外 3 个是 no-op。

#### 3. 消息 [5] 跳到 [6] 的内部步骤

1. `ToolNode` 并发执行 4 个 `calculator_wstate`
2. 每个工具返回 `Command(update={"ops": [...], "messages": [ToolMessage(...)]})`
3. 框架按 **reducer** 把 4 个 update 合并进 state(`messages` ← `add_messages`、`ops` ← `reduce_list`)
4. `tools` 节点结束 → 走无条件边 → 触发 `agent` 节点
5. `agent` 节点把**整个 `state["messages"]`** 重放给 LLM
6. LLM 生成新的 AIMessage,即 message [6]

#### 4. 必须记住的几个协议要点

| 要点 | 说明 |
|---|---|
| `tool_call_id` 配对 | `AIMessage.tool_calls[i].id` 必须 = `ToolMessage.tool_call_id`,否则 LLM API 报错 |
| `content=''` 是正常的 | 当 LLM 只想调工具时,`AIMessage.content` 为空,信息全在 `tool_calls` 里 |
| `finish_reason` | `'tool_calls'` → 继续循环;`'stop'` → 退出 |
| LLM API 无状态 | 每轮把全部 messages 重发,`input_tokens` 单调上涨(192→328→363→399→436) |
| `Command` 接管返回后 | 框架不再自动包 ToolMessage,**必须自己造**并填 `tool_call_id` |

#### 5. 一句话总结

> "循环"不在 LLM 里,而在 **LangGraph 这台有限状态机**里。它在 `agent` ↔ `tools` 之间反复跳,每轮把更新过的 messages 整段重放给 LLM,直到某个 AIMessage 不再带 `tool_calls`,就走向 END。


### 📖 详细解读:`result4["messages"]` 的 13 条消息

我们用这份 raw 输出**逐条**对照,把 ReAct 循环看个透。一共 **13 条消息**,可以分成 5 个"段落"。

#### 整体结构

```
[0]  HumanMessage           ← 用户输入
[1]  AIMessage (4 tool_calls)
[2]  ToolMessage             ┐
[3]  ToolMessage             │ 第 1 轮(并行 4 个)
[4]  ToolMessage             │
[5]  ToolMessage             ┘
[6]  AIMessage (1 tool_call)
[7]  ToolMessage             ← 第 2 轮:multiply
[8]  AIMessage (1 tool_call)
[9]  ToolMessage             ← 第 3 轮:add
[10] AIMessage (1 tool_call)
[11] ToolMessage             ← 第 4 轮:divide
[12] AIMessage (final)       ← 没有 tool_calls,循环结束
```

---

#### 段落一:用户输入(消息 0)

```python
HumanMessage(content='Please calculate ((15+23)x64+52)/6?', id='01d61d8a-...')
```
就是用户那句话被包成了 `HumanMessage`,加上一个自动生成的 `id`。

---

#### 段落二:第 1 轮并行调用(消息 1–5)

**消息 1:AIMessage,发出 4 个 tool_calls**

```python
AIMessage(
    content='',                           # ⚠️ 注意:文本内容是空的
    tool_calls=[
        {'name': 'calculator_wstate', 'args': {'operation': 'add',      'a': 15, 'b': 23}, 'id': 'call_dFGuO...'},
        {'name': 'calculator_wstate', 'args': {'operation': 'multiply', 'a': 64, 'b': 1},  'id': 'call_PEGsu...'},
        {'name': 'calculator_wstate', 'args': {'operation': 'add',      'a': 52, 'b': 0},  'id': 'call_KLpA4...'},
        {'name': 'calculator_wstate', 'args': {'operation': 'divide',   'a': 6,  'b': 1},  'id': 'call_NmpSa...'},
    ],
    response_metadata={
        'finish_reason': 'tool_calls',    # 🔑 关键:告诉 graph 还要继续循环
        ...
    },
    usage_metadata={'input_tokens': 192, 'output_tokens': 112},
)
```

要看的几个点:
- **`content=''`**:LLM 这一步没写自然语言,只发了 tool calls。这是 ReAct 中"行动"的状态。
- **`finish_reason='tool_calls'`**:OpenAI API 用这个字段告诉外层"我停下来是因为要调用工具,不是说完了"。LangGraph 的 ReAct 节点正是看这个(以及 `tool_calls` 字段是否非空)来决定走 `tools` 节点而不是 `END`。
- **`tool_calls` 是一个 list**:四个 dict 平铺在一起 → 这就是**并行调用**的物理表现。`ToolNode` 看到 4 个,就并发执行 4 个。
- **`input_tokens=192`**:此时上下文只有 system_prompt + user message + tool schema。

**消息 2–5:4 条 ToolMessage(对应 4 个并行结果)**

```python
ToolMessage(content='38',  name='calculator_wstate', tool_call_id='call_dFGuO...')  # ← 对应 add(15,23)
ToolMessage(content='64',  name='calculator_wstate', tool_call_id='call_PEGsu...')  # ← 对应 multiply(64,1)
ToolMessage(content='52',  name='calculator_wstate', tool_call_id='call_KLpA4...')  # ← 对应 add(52,0)
ToolMessage(content='6.0', name='calculator_wstate', tool_call_id='call_NmpSa...')  # ← 对应 divide(6,1)
```

**核心观察**:每条 `ToolMessage` 的 `tool_call_id` **必须**等于消息 1 里某个 `tool_call` 的 `id`。这是 LLM 把"我刚刚发了 4 个调用 → 我现在看到 4 个结果"对应起来的唯一办法。

> 还记得 `calculator_wstate` 工具代码吗?里面写的 `ToolMessage(f"{result}", tool_call_id=tool_call_id)` 就是为了**手动**填这个字段。如果你忘了填,LLM 就匹配不上,后面会乱。

---

#### 段落三:第 2 轮(消息 6–7)

**消息 6:AIMessage,1 个 tool_call**

```python
AIMessage(
    content='',
    tool_calls=[{'name': 'calculator_wstate', 'args': {'operation': 'multiply', 'a': 38, 'b': 64}, 'id': 'call_G2BUU...'}],
    response_metadata={'finish_reason': 'tool_calls', ...},
    usage_metadata={'input_tokens': 328, 'output_tokens': 24},
)
```

- LLM 现在看到第 1 轮的 4 个 ToolMessage 在历史里(`38, 64, 52, 6.0`),决定走真正的乘法。
- **`input_tokens` 涨到 328**(从 192 → 328):因为整个 messages 列表都会重新喂给 LLM。这是 ReAct 的一个"代价":每轮都要带着前面所有消息回去。

**消息 7:ToolMessage**

```python
ToolMessage(content='2432', tool_call_id='call_G2BUU...')
```

---

#### 段落四:第 3 轮(消息 8–9)

```python
AIMessage(tool_calls=[{... 'operation': 'add', 'a': 2432, 'b': 52, 'id': 'call_XB9qP...'}], ...)
ToolMessage(content='2484', tool_call_id='call_XB9qP...')
```
`input_tokens` 涨到 363。

---

#### 段落五:第 4 轮 + 结束(消息 10–12)

```python
AIMessage(tool_calls=[{... 'operation': 'divide', 'a': 2484, 'b': 6, 'id': 'call_ZORqs...'}], ...)
ToolMessage(content='414.0', tool_call_id='call_ZORqs...')
```
`input_tokens` 涨到 399。

**消息 12:最终 AIMessage(✅ 循环结束)**

```python
AIMessage(
    content='The result of the calculation ((15+23)x64+52)/6 is 414.0.',  # ← 这次有内容了
    tool_calls=[],                                                          # ← 🔑 空列表
    response_metadata={
        'finish_reason': 'stop',                                            # ← 🔑 不再是 'tool_calls'
        ...
    },
    usage_metadata={'input_tokens': 436, 'output_tokens': 22},
)
```

**这两个字段的对比是 ReAct 的退出条件**:

| 字段 | 之前的 AIMessage | 最终的 AIMessage |
|---|---|---|
| `content` | `''` | 自然语言答案 |
| `tool_calls` | 非空列表 | `[]` |
| `finish_reason` | `'tool_calls'` | `'stop'` |

LangGraph 的 ReAct 路由器就是看这几个字段判断:**"AIMessage 没有 tool_calls → 走向 END,循环结束"**。

---

#### 一个你应该留意的"成本信号":input_tokens 的增长

```
Round 1 input_tokens: 192   (system + user + tool schemas)
Round 2 input_tokens: 328   (+ 第1轮的 AIMessage + 4 个 ToolMessage)
Round 3 input_tokens: 363
Round 4 input_tokens: 399
Final  input_tokens: 436
```

每多一轮,**整个消息历史都会重新喂给 LLM**。这就是为什么:
- 当 messages 很长时,token 成本会快速膨胀;
- 后面的课程会引入**子代理(subagent)** 和**文件系统(virtual filesystem)**,目的就是把不需要每次都看的信息**搬出主对话**,只在主代理 messages 里留摘要 —— 这叫 **context offloading / context quarantine**。

---

#### 一行图总结消息流

```
H  → AI[4 calls]  → T,T,T,T  → AI[1 call] → T  → AI[1 call] → T  → AI[1 call] → T  → AI[content, no calls = END]
 0       1          2 3 4 5        6         7       8         9       10        11           12
```

`H` = HumanMessage,`AI` = AIMessage,`T` = ToolMessage。每个 `AI[...]` 是一次 LLM 调用,每段 `T...T` 是 ToolNode 的一次执行。
